Example Simple Chat bot Gateway

In [40]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import os
import sqlite3

In [41]:
load_dotenv(override=True)
openai = OpenAI()


In [42]:
#code to functions to search ind.db to register as a tool for chat agent
# Method to search by indicator name
def search_ind_name(ind_name: str) -> list | None:
    """ Get the list of URLS for given question about indicators """
    conn = sqlite3.connect("intro/ind.db")
    c = conn.cursor()
    # Use wildcards for partial matching and case-insensitive search
    search_pattern = f"%{ind_name.lower()}%"
    c.execute("SELECT url FROM indicators WHERE LOWER(name) LIKE ?", (search_pattern,))
    result = c.fetchall()
    conn.close()
    return result if result else None

# Enhanced search function with more options
def search_indicators_advanced(search_term: str, exact_match: bool = False) -> list | None:
    """ Advanced search with options for exact or partial matching """
    conn = sqlite3.connect("intro/ind.db")
    c = conn.cursor()
    
    if exact_match:
        # Exact match (case-insensitive)
        c.execute("SELECT name, url FROM indicators WHERE LOWER(name) = LOWER(?)", (search_term,))
    else:
        # Partial match with wildcards
        search_pattern = f"%{search_term.lower()}%"
        c.execute("SELECT name, url FROM indicators WHERE LOWER(name) LIKE ?", (search_pattern,))
    
    result = c.fetchall()
    print(str(result.count))
    conn.close()
    return result if result else None

# Function to get all indicators
def get_all_indicators() -> list:
    """ Get all indicators from the database """
    conn = sqlite3.connect("intro/ind.db")
    c = conn.cursor()
    c.execute("SELECT name, url FROM indicators ORDER BY name")
    result = c.fetchall()
    conn.close()
    return result

In [43]:
gw_tools = [search_indicators_advanced]

In [44]:
with open("intro/euro_countries.json", "r", encoding="utf-8") as f:
    euro_countries = f.read()

with open("intro/euro_cntry_grp.json", "r", encoding="utf-8") as f:
    euro_cntry_groups = f.read()
    
with open("intro/cntry_grp_map.csv", "r", encoding="utf-8") as f:
    cntry_grp_map = f.read()

In [45]:
with open("intro/about.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [46]:
agent_name = "European Health Information Gateway"

In [47]:
# create model client autogen
from autogen_ext.models.openai import OpenAIChatCompletionClient
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

In [48]:
system_prompt = f"You are acting as {agent_name}. You are answering questions on {agent_name}'s data portal, \
particularly questions related to {agent_name}'s health data, background, provide the links to health indicators from gateway.euro.who.int. \
Your responsibility is to represent {agent_name} for interactions on the website as faithfully as possible. \
    You task is to  guides users around gateway.euro.who.int and also helps them analyze and compare data \
You are given a table of {agent_name}'s indicators and summary about the portal which you can use to answer questions. \
    In addition  you will have country group list and you will need to inform that in graphs they can select countries and groups of countries \
Be professional and engaging, as if talking to data health scientist or a usual person who came across the website and interested in health data. \
If you don't know the answer, say so. \
IMPORTANT: ONLY provide URLs that are retrieved from the database using the search functions. Do NOT provide any URLs from your training data as they may be outdated. Always use the search_ind_name, search_indicators_advanced, or get_all_indicators functions to get current, valid URLs before sharing any links with users."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## health indicators list as sqlite database:please respond with links only from provided list\n\n \
## list of countries in WHO Europe in JSON:\n{euro_countries}\n\n \
## list of countries groups WHO Europe in JSON:\n{euro_cntry_groups}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {agent_name}."

In [49]:
system_prompt

'You are acting as European Health Information Gateway. You are answering questions on European Health Information Gateway\'s data portal, particularly questions related to European Health Information Gateway\'s health data, background, provide the links to health indicators from gateway.euro.who.int. Your responsibility is to represent European Health Information Gateway for interactions on the website as faithfully as possible.     You task is to  guides users around gateway.euro.who.int and also helps them analyze and compare data You are given a table of European Health Information Gateway\'s indicators and summary about the portal which you can use to answer questions.     In addition  you will have country group list and you will need to inform that in graphs they can select countries and groups of countries Be professional and engaging, as if talking to data health scientist or a usual person who came across the website and interested in health data. If you don\'t know the answe

In [50]:
from autogen_agentchat.agents import AssistantAgent

smart_agent = AssistantAgent(
    name='gw_chat',
    model_client=model_client,
    system_message= system_prompt,
    model_client_stream=True,
    tools=gw_tools,
    reflect_on_tool_use=True
)

In [51]:
import asyncio
from autogen_core import CancellationToken
from autogen_agentchat.messages import TextMessage

def chat(message, history):
    """Chat function that properly handles the smart agent in Jupyter environment"""
    try:
        # Create a new event loop in a separate thread to avoid conflicts with Jupyter's loop
        import concurrent.futures
        import threading
        
        def run_async():
            # Create a new event loop for this thread
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            try:
                return loop.run_until_complete(chat_async(message, history))
            finally:
                loop.close()
        
        # Run the async function in a separate thread
        with concurrent.futures.ThreadPoolExecutor() as executor:
            future = executor.submit(run_async)
            return future.result()
            
    except Exception as e:
        return f"Error: {str(e)}"

async def chat_async(message, history):
    """Async chat function that properly handles the smart agent"""
    try:
        # Convert string message to TextMessage object that autogen expects
        text_message = TextMessage(content=message, source="user")
        response = await smart_agent.on_messages([text_message], cancellation_token=CancellationToken())
        # Access the content from the response
        return response.chat_message.content
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
# Launch the Gradio chat interface with proper message format
interface = gr.ChatInterface(
    fn=chat,
    title="European Health Information Gateway Chat",
    description="Ask questions about health indicators and data from the WHO European Region",
    examples=[
        "What health indicators are available?",
        "Show me data about life expectancy",
        "What countries are in the Nordic group?",
        "How can I access the health data?"
    ],
    type="messages"  # Use the new message format to avoid deprecation warning
)

interface.launch(share=False)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Task was destroyed but it is pending!
task: <Task pending name='Task-5930' coro=<<async_generator_athrow without __name__>()>>
